# Adding a New Language to the LID Model

This notebook extends the model trained in **`train-and-evaluate-local-lid.ipynb`** by:

1. Downloading a Mozilla Data Collective corpus for a language **not** in the original CV-LID training data.
2. Incorporating that corpus into the training data and re-fitting the model.
3. Evaluating the updated model on examples from a **second MDC dataset** in that language, and comparing performance before and after the update.

All data comes from the [Mozilla Data Collective](https://datacollective.org).

The target language is **Gujarati (`guj`)**, an Indo-Aryan language spoken primarily in the Indian state of Gujarat, which is absent from CV-LID training data but present in CommonLID — making it a clean before/after test case. So, we want to see, **by adding some text from a Gujarati corpus on MDC, how much better does our CV-LID model perform on Gujarati in the CommonLID task?**

To add a different language, update `TARGET_LANG`, `TRAIN_DATASET_ID`, and `EVAL_DATASET_ID` at the top of **Section 1**.

In [3]:
import os
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
from datacollective import download_dataset
from fox_progress_bar import ProgressBar

# utility functions
from language_id.data import (load_commonlid, COMMON_VOICE_LID_DATASET_ID)
from language_id.lang_codes_mapping import to_iso3

## Configuration

Set the target language and the two MDC dataset IDs below, then run the rest of the notebook.

- **`TRAIN_DATASET_ID`** — an MDC dataset used to *train* the model on the new language.  
  Here we use the Gujarati corpus (`cmp8vqj5h03m0o007qu438s4a`) from MDC.
- **`EVAL_DATASET_ID`** — a *separate* MDC dataset used only for evaluation, so results are not trivially easy.  
  Here we use **CommonLID** itself, which already has Gujarati examples and is an independent Common Crawl–derived corpus.

In [15]:
%env MDC_API_KEY=<YOUR MDC API KEY HERE>

env: MDC_API_KEY=<YOUR MDC API KEY HERE>


In [6]:
# ── Target language (ISO 639-3) ────────────────────────────────────────────────
TARGET_LANG = "guj"   # Gujarati

# ── MDC dataset IDs ────────────────────────────────────────────────────────────
# Training corpus: the Gujarati dataset on MDC (dataset ID provided).
TRAIN_DATASET_ID = "cmp8vqj5h03m0o007qu438s4a"   # Gujarati training corpus

sample_p = 0.15
SEED = 42

cv_lid_path = download_dataset(COMMON_VOICE_LID_DATASET_ID)
parent_dir = str(cv_lid_path).rsplit("/", 1)[0]
cv_lid_untarred_path = (parent_dir
                        + "/"
                        + "mozilla-common-voice-text-language-ident-b1b3aae0")

if not os.path.exists(cv_lid_untarred_path):
    os.system(f"mkdir {cv_lid_untarred_path}")
    os.system(f"tar -xzf {cv_lid_path} -C {cv_lid_untarred_path}")

path_to_file = cv_lid_untarred_path + "/mcv_text_lid/mcv_text_lid.tsv"


def get_split_file_path(split):
    return f"{cv_lid_untarred_path}/mcv_text_lid/" + f"{split}_sample.tsv"

for split in ("train", "test", "dev"):
    if os.path.exists(f"{get_split_file_path(split)}"):
        os.system(f"rm {get_split_file_path(split)}")

print("Processing 20 million row files from disk to create sampled train, dev, and test files.")
rows_processed = 0
pb = ProgressBar(total_size=19750000, unit="Rows")
for chunk in pd.read_csv(path_to_file, sep="\t", chunksize=100_000):
    for split, grp in chunk.groupby("split"):
        path_to_split_file = get_split_file_path(split)
        grp.sample(frac=sample_p, random_state=SEED).to_csv(
            path_to_split_file, sep="\t", mode="a", index=False,
            header=not os.path.exists(path_to_split_file)
        )
    pb.update(100_000)

cv_lid_train = pd.read_csv(get_split_file_path("train"), sep="\t")
cv_lid_dev = pd.read_csv(get_split_file_path("dev"), sep="\t")
cv_lid_test = pd.read_csv(get_split_file_path("test"), sep="\t")

cv_lid_train["lang"] = cv_lid_train["lang"].astype(str).map(to_iso3)
cv_lid_dev["lang"] = cv_lid_dev["lang"].astype(str).map(to_iso3)
cv_lid_test["lang"] = cv_lid_test["lang"].astype(str).map(to_iso3)

print("Finished.")


Processing 20 million row files from disk to create sampled train, dev, and test files.
█████████████████████████████████████████████████🦊 100.0% (19800000.0 Rows/19750000.0 Rows) 511400.0 Rows/s ETA: --:--Finished.


## Confirm the language is absent from the original training data

In [7]:
n_existing = (cv_lid_train["lang"] == TARGET_LANG).sum()
print(f"Existing training examples for '{TARGET_LANG}': {n_existing:,}")
if n_existing > 0:
    print("  ⚠ Language already present — this notebook will add MORE data for it.")
else:
    print("  ✓ Language is absent from CV-LID — we will bootstrap it from scratch.")

Existing training examples for 'guj': 0
  ✓ Language is absent from CV-LID — we will bootstrap it from scratch.


## Download the training corpus from MDC

You can download via the Python SDK and point this code to that location

In [8]:
import glob
path_to_downloaded_dataset = "/home/rob/Downloads/Gujarati Text Corpus"

all_guj_text = []
for fname in glob.glob(path_to_downloaded_dataset + "/*/*/*txt"):
    with open(fname) as f:
        txt = f.read().split("\n")
        all_guj_text.extend(txt)
len(all_guj_text)

import random
guj_training_txt = random.sample(all_guj_text, 50000)
guj_df = pd.DataFrame({"lang": ["guj"] * len(guj_training_txt), "sentence": guj_training_txt})

In [9]:
updated_cvlid_training_data = pd.concat([cv_lid_train[["sentence", "lang"]], guj_df])


## Download the evaluation corpus from MDC

We use **CommonLID** — already downloaded in the prerequisite notebook — as our held-out evaluation source.  CommonLID is derived from Common Crawl (a completely different pipeline from Common Voice), so it provides a genuine out-of-domain test.

If CommonLID does not cover the target language, swap `EVAL_DATASET_ID` for any other MDC dataset that does (e.g. a different Common Voice split or a dedicated text corpus).

In [10]:
commonlid_df = load_commonlid()

## Re-fit the model


In [11]:
vectorizer = HashingVectorizer(
    analyzer='char_wb',      # character n-grams padded at word boundaries
    ngram_range=(2, 4),      # bigrams, trigrams, and 4-grams
    n_features=2**18,        # ~262k feature buckets...you can play with this...
    alternate_sign=False,    # keep values non-negative
    norm=None,               # raw counts, not L2-normalized
)

In [12]:
clf = MultinomialNB(alpha=0.01)

In [13]:
X = vectorizer.transform(updated_cvlid_training_data["sentence"])
print("Transformed training text into features...")
y = updated_cvlid_training_data["lang"]
print("Fitting model...")
clf.fit(X, y)
print("Training complete.")

Transformed training text into features...
Fitting model...
Training complete.


## 6. Evaluate — before vs. after adding the new language
Previous score was 0 since CV LID never saw Gujarati.

Now let's see how well it does now that we've added some `guj` data to the training corpus:

In [14]:
X_eval = vectorizer.transform(commonlid_df["sentence"])
y_true = np.array(commonlid_df["lang"])
y_pred  = clf.predict(X_eval)

print(classification_report(y_true, y_pred, digits=3, zero_division=0, labels=["guj"]))

              precision    recall  f1-score   support

         guj      0.955     0.953     0.954       948

   micro avg      0.955     0.953     0.954       948
   macro avg      0.955     0.953     0.954       948
weighted avg      0.955     0.953     0.954       948

